FEATURE ENGINEERING & ANALYSIS

In [1]:
import pandas as pd
data_dir = '../data/'


messages = pd.read_csv(data_dir+'messages.csv')
messages = messages.sort_values(['user_id', 'session_type', 'message_id'])

# Shift to get the previous message's role and timestamp
messages['prev_role'] = messages.groupby(['user_id', 'session_type'])['role'].shift(1)
messages['prev_timestamp'] = messages.groupby(['user_id', 'session_type'])['timestamp'].shift(1)

# Calculate time since previous message
messages['time_since_prev'] = messages['timestamp'] - messages['prev_timestamp']

user_responses = messages[messages['role'] == 'user'].copy()

user_responses['time_since_prev'] = (
    user_responses['timestamp'] - user_responses['prev_timestamp']
)
print("User responses shape:", user_responses.shape)
print(user_responses[['role','prev_role','time_since_prev']].head())
print("Any NaNs in time_since_prev:", user_responses['time_since_prev'].isna().mean())



response_times = user_responses.groupby(['user_id', 'session_type']).agg(
    avg_response_time=('time_since_prev', 'mean'),
    median_response_time=('time_since_prev', 'median'),
    std_response_time=('time_since_prev', 'std'),
    min_response_time=('time_since_prev', 'min'),
    max_response_time=('time_since_prev', 'max')
).reset_index()
print("Response_times shape:", response_times.shape)
print(response_times.head())

user_responses['rapid_response'] = user_responses['time_since_prev'] < 10

rapid_features = user_responses.groupby(['user_id', 'session_type']).agg(
    rapid_response_count=('rapid_response', 'sum'),
    rapid_response_pct=('rapid_response', 'mean')  # proportion of responses that were quick, possibly didn't read the prompt
).reset_index()

User responses shape: (1390, 10)
   role  prev_role  time_since_prev
1  user  assistant        15.972857
3  user  assistant        43.338320
5  user  assistant        32.207580
7  user  assistant        21.481455
9  user  assistant        35.729019
Any NaNs in time_since_prev: 0.0035971223021582736
Response_times shape: (114, 7)
                        user_id session_type  avg_response_time  \
0  1N7UGUGuQHhcaUyk5AnPXTVxsDd2    arraylist          37.021334   
1  1N7UGUGuQHhcaUyk5AnPXTVxsDd2    recursion          28.231354   
2  2cjI20Jq7PYlvEozFwC9A1koWYG3    arraylist          54.091446   
3  2cjI20Jq7PYlvEozFwC9A1koWYG3    recursion          48.660198   
4  2gu7Ew3WFuME2poY733nrLDG6oh2    arraylist          21.850887   

   median_response_time  std_response_time  min_response_time  \
0             26.266255          22.276396          15.741244   
1             23.446072          18.587822           6.089680   
2             35.909532          46.313270          12.320833   
3     

In [2]:
messages = pd.read_csv(data_dir + "messages.csv")
sessions = pd.read_csv(data_dir + "sessions.csv")
sessions_complete = sessions[sessions['status'] == 'completed'].copy()
msg_counts = messages.groupby(['user_id','session_type']).agg(
    total_messages = ('message_id', 'count'),
    user_messages = ('role', lambda x: (x == 'user').sum()),
    assistant_messages = ('role', lambda x: (x == 'assistant').sum())
).reset_index()
messages = messages.sort_values(['user_id','session_type','timestamp'])




In [3]:
messages['prev_timestamp'] = messages.groupby(['user_id','session_type'])['timestamp'].shift(1)
messages['inter_message_time'] = messages['timestamp'] - messages['prev_timestamp']
pacing = messages.groupby(['user_id','session_type']).agg(
    median_inter_message_time = ('inter_message_time', 'median'),
    std_inter_message_time = ('inter_message_time', 'std'),
    proportion_long_gaps = ('inter_message_time', lambda x: (x > 60).mean()),
    proportion_short_bursts = ('inter_message_time', lambda x: (x < 10).mean())
).reset_index()


In [4]:
messages['msg_length'] = messages['content'].str.len()

user_msg_features = messages[messages['role'] == 'user'].groupby(['user_id','session_type']).agg(
    avg_user_message_length = ('msg_length', 'mean'),
    median_user_message_length = ('msg_length', 'median'),
    proportion_long_messages = ('msg_length', lambda x: (x > 120).mean())
).reset_index()


In [5]:
messages['prev_role'] = messages.groupby(['user_id','session_type'])['role'].shift(1)

interaction = messages.groupby(['user_id','session_type']).agg(
    back_and_forth_count = ('role', lambda x: (x.shift(1) != x).sum()),
    user_to_assistant_ratio = ('role', lambda x: (x == 'user').sum() / max((x == 'assistant').sum(), 1))
).reset_index()


In [6]:

session_df = sessions_complete.copy()
print(session_df.columns)
session_df = session_df.merge(msg_counts, on=['user_id','session_type'], how='left')
session_df = session_df.merge(pacing, on=['user_id','session_type'], how='left')
session_df = session_df.merge(user_msg_features, on=['user_id','session_type'], how='left')
session_df = session_df.merge(interaction, on=['user_id','session_type'], how='left')

session_df.shape
print(session_df.columns.tolist())


Index(['user_id', 'session_type', 'status', 'condition', 'start_time',
       'end_time', 'duration_seconds', 'total_messages', 'user_messages',
       'assistant_messages', 'quiz_score', 'quiz_total', 'quiz_percentage',
       'quiz_completed_time', 'survey_completed_time',
       'avg_difficulty_correct', 'avg_difficulty_incorrect'],
      dtype='object')
['user_id', 'session_type', 'status', 'condition', 'start_time', 'end_time', 'duration_seconds', 'total_messages_x', 'user_messages_x', 'assistant_messages_x', 'quiz_score', 'quiz_total', 'quiz_percentage', 'quiz_completed_time', 'survey_completed_time', 'avg_difficulty_correct', 'avg_difficulty_incorrect', 'total_messages_y', 'user_messages_y', 'assistant_messages_y', 'median_inter_message_time', 'std_inter_message_time', 'proportion_long_gaps', 'proportion_short_bursts', 'avg_user_message_length', 'median_user_message_length', 'proportion_long_messages', 'back_and_forth_count', 'user_to_assistant_ratio']


In [7]:
session_df['total_messages'] = session_df['total_messages_y']
session_df['user_messages'] = session_df['user_messages_y']
session_df['assistant_messages'] = session_df['assistant_messages_y']

session_df['messages_per_minute'] = session_df['total_messages'] / (session_df['duration_seconds'] / 60)

session_df = session_df.drop(columns=[
    'total_messages_x', 'user_messages_x', 'assistant_messages_x',
    'total_messages_y', 'user_messages_y', 'assistant_messages_y'
])


In [8]:

# Merge into sessions
sessions_complete = sessions_complete.merge(response_times, on=['user_id', 'session_type'], how='left')

sessions_complete = sessions_complete.merge(rapid_features, on=['user_id', 'session_type'], how='left')
print("Sessions_complete pacing NA rates:")
print(
    sessions_complete.groupby('condition')[
        ['avg_response_time','std_response_time','max_response_time']
    ].apply(lambda x: x.isna().mean())
)

sessions_complete.head()


Sessions_complete pacing NA rates:
           avg_response_time  std_response_time  max_response_time
condition                                                         
1.0                 0.024390           0.024390           0.024390
2.0                 0.051282           0.051282           0.051282
3.0                 0.000000           0.000000           0.000000


,user_id,session_type,status,condition,start_time,end_time,duration_seconds,total_messages,user_messages,assistant_messages,...,survey_completed_time,avg_difficulty_correct,avg_difficulty_incorrect,avg_response_time,median_response_time,std_response_time,min_response_time,max_response_time,rapid_response_count,rapid_response_pct
0,1N7UGUGuQHhcaUyk5AnPXTVxsDd2,arraylist,completed,3.0,1.769195e+09,1.769196e+09,1009.064499,37.0,18.0,19.0,...,1.769196e+09,2.40,0.00,37.021334,26.266255,22.276396,15.741244,98.225849,0.0,0.000000
1,1N7UGUGuQHhcaUyk5AnPXTVxsDd2,recursion,completed,3.0,1.769799e+09,1.769800e+09,969.275643,47.0,24.0,23.0,...,1.769800e+09,2.33,2.00,28.231354,23.446072,18.587822,6.089680,84.877592,2.0,0.083333
2,2cjI20Jq7PYlvEozFwC9A1koWYG3,arraylist,completed,2.0,1.769191e+09,1.769192e+09,647.387764,17.0,8.0,9.0,...,1.769192e+09,2.40,0.00,54.091446,35.909532,46.313270,12.320833,148.132002,0.0,0.000000
3,2cjI20Jq7PYlvEozFwC9A1koWYG3,recursion,completed,2.0,1.769796e+09,1.769796e+09,666.703658,23.0,11.0,12.0,...,1.769796e+09,2.00,3.00,48.660198,31.064684,38.848969,5.821363,118.538725,1.0,0.090909
4,2gu7Ew3WFuME2poY733nrLDG6oh2,arraylist,completed,3.0,1.769188e+09,1.769189e+09,981.592234,65.0,32.0,33.0,...,1.769189e+09,3.50,1.67,21.850887,17.835211,17.537275,2.820791,69.048597,9.0,0.281250


In [9]:
print(sessions_complete.shape)


(115, 24)


In [10]:
# determine if there are sessions that don't have both arraylist and recursion
session_counts = sessions_complete.groupby(['user_id', 'session_type']).size().unstack(fill_value=0)
missing = session_counts[(session_counts['arraylist'] == 0) | (session_counts['recursion'] == 0)]
missing
sessions_missing = sessions_complete[sessions_complete['user_id'].isin(missing.index)]
sessions_missing

# First compute which users have both
users_with_both = session_counts[(session_counts['arraylist'] > 0) & (session_counts['recursion'] > 0)].index

# Add a column to the main df
sessions_complete['has_both'] = sessions_complete['user_id'].isin(users_with_both)
print(sessions_complete.head())
sessions_complete.to_csv(data_dir + 'sessions_with_engagement_features.csv', index=False)



                        user_id session_type     status  condition  \
0  1N7UGUGuQHhcaUyk5AnPXTVxsDd2    arraylist  completed        3.0   
1  1N7UGUGuQHhcaUyk5AnPXTVxsDd2    recursion  completed        3.0   
2  2cjI20Jq7PYlvEozFwC9A1koWYG3    arraylist  completed        2.0   
3  2cjI20Jq7PYlvEozFwC9A1koWYG3    recursion  completed        2.0   
4  2gu7Ew3WFuME2poY733nrLDG6oh2    arraylist  completed        3.0   

     start_time      end_time  duration_seconds  total_messages  \
0  1.769195e+09  1.769196e+09       1009.064499            37.0   
1  1.769799e+09  1.769800e+09        969.275643            47.0   
2  1.769191e+09  1.769192e+09        647.387764            17.0   
3  1.769796e+09  1.769796e+09        666.703658            23.0   
4  1.769188e+09  1.769189e+09        981.592234            65.0   

   user_messages  assistant_messages  ...  avg_difficulty_correct  \
0           18.0                19.0  ...                    2.40   
1           24.0                23.0  

In [11]:
# Fix the quiz level ratings
quiz = pd.read_csv(data_dir + 'quiz_questions.csv')
quiz['question_number'].unique()
# Fix 0-indexed question numbers
quiz['question_number'] = quiz['question_number'] + 1
arraylist_difficulty_map = {
    1: 1,
    2: 2,
    3: 2,
    4: 2,
    5: 4
}

recursion_difficulty_map = {
    1: 1,
    2: 2,
    3: 3,
    4: 1,
    5: 3,
    6: 4,
    7: 3
}

quiz['difficulty'] = quiz.apply(
    lambda row: arraylist_difficulty_map[row['question_number']]
                if row['session_type'] == 'arraylist'
                else recursion_difficulty_map[row['question_number']],
    axis=1
)


difficulty_features = quiz.groupby(['user_id','session_type']).apply(
    lambda g: pd.Series({
        'avg_difficulty_correct': g.loc[g['is_correct']==1, 'difficulty'].mean(),
        'avg_difficulty_incorrect': g.loc[g['is_correct']==0, 'difficulty'].mean()
    })
).reset_index()


/var/folders/5w/yxkxqv7j6cn7rlnwhn2dhhl80000gn/T/ipykernel_99324/3163494475.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  difficulty_features = quiz.groupby(['user_id','session_type']).apply(


In [12]:
sessions_complete = sessions_complete.drop(
    columns=['avg_difficulty_correct','avg_difficulty_incorrect'],
    errors='ignore'
)

sessions_complete = sessions_complete.merge(
    difficulty_features,
    on=['user_id','session_type'],
    how='left'
)

quiz.to_csv(data_dir + "quiz_questions_updated.csv", index=False)
sessions_complete.to_csv(data_dir + "sessions_with_engagement_features_updated.csv", index=False)



In [13]:
sessions_complete.columns

Index(['user_id', 'session_type', 'status', 'condition', 'start_time',
       'end_time', 'duration_seconds', 'total_messages', 'user_messages',
       'assistant_messages', 'quiz_score', 'quiz_total', 'quiz_percentage',
       'quiz_completed_time', 'survey_completed_time', 'avg_response_time',
       'median_response_time', 'std_response_time', 'min_response_time',
       'max_response_time', 'rapid_response_count', 'rapid_response_pct',
       'has_both', 'avg_difficulty_correct', 'avg_difficulty_incorrect'],
      dtype='object')